## Import Libraries

In [2]:
import pandas as pd 
import numpy as np
import seaborn as sns 
import matplotlib.pyplot as plt

## Load Dataset 

In [3]:
df = pd.read_csv("cleaned_amazon.csv")

## Data Cleaning

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 661 entries, 0 to 660
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Title            661 non-null    object 
 1   ratings          661 non-null    float64
 2   total_reviews    661 non-null    int64  
 3   buying_history   661 non-null    object 
 4   alinknormal_URL  661 non-null    object 
 5   Price            661 non-null    float64
 6   Image            661 non-null    object 
dtypes: float64(2), int64(1), object(4)
memory usage: 36.3+ KB


In [5]:
df.isnull().sum()

Title              0
ratings            0
total_reviews      0
buying_history     0
alinknormal_URL    0
Price              0
Image              0
dtype: int64

In [6]:
df.head()

,Title,ratings,total_reviews,buying_history,alinknormal_URL,Price,Image
0,Plant-Based Kitchen Sponges - FSC Certified an...,4.6,248,3K+ bought in past month,https://www.amazon.com/Isshah-Plant-Based-Kitc...,9.99,https://m.media-amazon.com/images/I/81GEZUgvk2...
1,Reusable Paper Towels Washable Roll - Thick 12...,4.6,205,100+ bought in past month,https://www.amazon.com/EcoPledge-Reusable-Pape...,24.99,https://m.media-amazon.com/images/I/91-W1NYNeY...
2,20 Pack Reusable Paper Towels Washable - Natur...,4.4,3274,900+ bought in past month,https://www.amazon.com/Reusable-Alternative-Wa...,31.94,https://m.media-amazon.com/images/I/81WV3GCPce...
3,Akeeko Reusable Beeswax Food Wraps - Assorted ...,4.4,2426,50+ bought in past month,https://www.amazon.com/Reusable-Wraps-Beeswax-...,13.99,https://m.media-amazon.com/images/I/91UpDUvsk-...
4,Swedish Dishcloths for Kitchen Dish Towels - 6...,4.6,1286,100+ bought in past month,https://www.amazon.com/Harps-Seb-Reusable-Swed...,13.90,https://m.media-amazon.com/images/I/91FngmRdt1...


In [7]:
categorial_cols = df.select_dtypes(include="object").columns

for col in categorial_cols:
    print("\n---",col,"---")
    print(df[col].value_counts().head(20))


--- Title ---
Title
Reusable Beeswax Wrap - 9 Pack Beeswax Wraps For Food, Organic, Sustainable, Biodegradable, Zero Waste, Plastic-Free Food Storage, 1L Strawberry, 3M Orange, 5S Lemon Patterns                            6
Tieralia 5-Piece Eco-Friendly Bamboo Dish Brush Set with Compostable Wood Pulp Sponges | Bamboo Kitchen Cleaning Set                                                                                      6
Plant-Based Kitchen Sponges - FSC Certified and PETA Approved, Natural, Eco-Friendly, Biodegradable Sisal Hemp Sponges for Dishes, Cleaning Sponge for Kitchen, Bathroom, Household - 12 Count            6
Bamboo Cutting Board, Durable Wood Cutting Boards for Kitchen with Deep Juice Grooves & Built-in Handles, Ideal Charcuterie & Chopping for Meat, Vegetables - Perfect Kitchen Gift for Home Cooks         5
Natural Kitchen Sponge - Biodegradable Compostable Cellulose and Coconut Scrubber Sponge - Pack of 12 Eco Friendly Sponges for Dishes                              

In [8]:
numerical_cols = df.select_dtypes(include=np.number).columns
print(numerical_cols.tolist())

['ratings', 'total_reviews', 'Price']


In [9]:
df_analysis = df.copy()

In [10]:
df_analysis["product_text"] = (
    df_analysis['Title']
    .fillna("")
    .astype(str)
    .str.lower()
)

## Sustainability Keyword Detection

In [19]:
sustainability_keywords = {
    "strong": [
        "biodegradable",
        "compostable",
        "zero waste",
        "plastic-free",
        "plastic free",
        "reusable"
    ],

    "medium": [
        "sustainable",
        "eco-friendly",
        "eco friendly",
        "organic",
        "plant-based",
        "plant based",
        "fsc certified",
        "bpi certified",
        "certified compostable",
        "ethical",
        "ethics",
        "slow fashion",
        "fair trade",
        "fairtrade",
        "responsibly made",
        "environmentally friendly",
        "enviroment friendly"
    ],

    "supporting": [
        "bamboo",
        "natural",
        "vegan",
        "washable",
        "recyclable",
    ]
}

In [12]:
keyword_weights = {
    "strong":3,
    "medium":2,
    "supporting":1
}

In [20]:
def find_sustainability_keywords(text):
    found = []
    
    for category, keywords in sustainability_keywords.items():
        for keyword in keywords:
            if keyword in text:
                found.append(keyword)
    
    return found

In [21]:
df_analysis["Found_Keywords"] = (
    df_analysis["product_text"]
    .apply(find_sustainability_keywords)
)

In [22]:
df_analysis[
    ["Title","Found_Keywords"]
].head(10)

,Title,Found_Keywords
0,Plant-Based Kitchen Sponges - FSC Certified an...,"[biodegradable, eco-friendly, plant-based, fsc..."
1,Reusable Paper Towels Washable Roll - Thick 12...,"[reusable, organic, washable]"
2,20 Pack Reusable Paper Towels Washable - Natur...,"[reusable, organic, washable]"
3,Akeeko Reusable Beeswax Food Wraps - Assorted ...,"[biodegradable, zero waste, plastic-free, reus..."
4,Swedish Dishcloths for Kitchen Dish Towels - 6...,"[biodegradable, reusable, washable]"
5,"Lucomb Swedish Dishcloths for Kitchen Dishes, ...","[biodegradable, reusable, eco friendly]"
6,Reusable Beeswax Wrap - 9 Pack Beeswax Wraps F...,"[biodegradable, zero waste, plastic-free, reus..."
7,100% Compostable Food Storage Bags [Quart 100 ...,"[compostable, reusable, eco-friendly, natural]"
8,Tieralia 5-Piece Eco-Friendly Bamboo Dish Brus...,"[compostable, eco-friendly, bamboo]"
9,Ultrasonic Pest Control Repeller - Repel Roden...,[eco-friendly]


In [27]:
def calculate_sustainability_score(text):
    score = 0
    for category, keywords in sustainability_keywords.items():
        for keyword in keywords:
            if keyword in text:
                score += keyword_weights[category]

    return score

## Sustainability Score

In [28]:
df_analysis["Sustainability_Score"] = (
    df_analysis["product_text"]
    .apply(calculate_sustainability_score)
)

In [29]:
df_analysis[
    ["Title","Found_Keywords","Sustainability_Score"]
].head(10)

,Title,Found_Keywords,Sustainability_Score
0,Plant-Based Kitchen Sponges - FSC Certified an...,"[biodegradable, eco-friendly, plant-based, fsc...",10
1,Reusable Paper Towels Washable Roll - Thick 12...,"[reusable, organic, washable]",6
2,20 Pack Reusable Paper Towels Washable - Natur...,"[reusable, organic, washable]",6
3,Akeeko Reusable Beeswax Food Wraps - Assorted ...,"[biodegradable, zero waste, plastic-free, reus...",18
4,Swedish Dishcloths for Kitchen Dish Towels - 6...,"[biodegradable, reusable, washable]",7
5,"Lucomb Swedish Dishcloths for Kitchen Dishes, ...","[biodegradable, reusable, eco friendly]",8
6,Reusable Beeswax Wrap - 9 Pack Beeswax Wraps F...,"[biodegradable, zero waste, plastic-free, reus...",16
7,100% Compostable Food Storage Bags [Quart 100 ...,"[compostable, reusable, eco-friendly, natural]",9
8,Tieralia 5-Piece Eco-Friendly Bamboo Dish Brus...,"[compostable, eco-friendly, bamboo]",6
9,Ultrasonic Pest Control Repeller - Repel Roden...,[eco-friendly],2


## Sustainability Evidence Level

In [32]:
def classify_sustainability(score):
    if score <=2:
        return "Low"
    elif score <= 5:
        return "Moderate"
    else:
        return "Strong"

df_analysis["Sustainability_Evidence_Level"]=(
    df_analysis["Sustainability_Score"]
    .apply(classify_sustainability)
)

In [33]:
df_analysis["Sustainability_Evidence_Level"].value_counts()

Sustainability_Evidence_Level
Low         349
Strong      166
Moderate    146
Name: count, dtype: int64

In [34]:
df_analysis["Sustainability_Score"].value_counts().sort_index()

Sustainability_Score
0     199
1      57
2      93
3      85
4      20
5      41
6      37
7      29
8      19
9      24
10     22
11      7
12      2
13      3
14     10
15      2
16      6
17      1
18      4
Name: count, dtype: int64

In [35]:
df_analysis["Found_Keywords"] =(
    df_analysis["product_text"]
    .apply(find_sustainability_keywords)
)

df_analysis["Sustainability_Score"] = (
    df_analysis["product_text"]
    .apply(calculate_sustainability_score)
)

df_analysis["Sustainability_Evidence_Level"] =(
    df_analysis["Sustainability_Score"]
    .apply(classify_sustainability)
)

In [36]:
df_analysis["Sustainability_Evidence_Level"].value_counts()

Sustainability_Evidence_Level
Low         349
Strong      166
Moderate    146
Name: count, dtype: int64

In [38]:
non_product_keywords = [
    "boook",
    "dummies",
    "handbook",
    "guide",
    "fashion ethics",
    "ethics",
    "how to",
    "history of",
    "the art of",
    "magazine"
]

def detect_non_product(title):
    title = str(title).lower()

    for keyword in non_product_keywords:
        if keyword in title:
            return "Possible Non-Product"
    return "Likely Product"
df_analysis["Product_Type"] = (
    df_analysis["Title"]
    .apply(detect_non_product)
)

In [39]:
df_analysis["Product_Type"].value_counts()

Product_Type
Likely Product          641
Possible Non-Product     20
Name: count, dtype: int64

In [40]:
df_analysis[
    df_analysis["Product_Type"] == "Possible Non-Product"
][
    ["Title","Sustainability_Score","Sustainability_Evidence_Level"]
].head(10)

,Title,Sustainability_Score,Sustainability_Evidence_Level
218,Waste-Free Kitchen Handbook: A Guide to Eating...,5,Moderate
281,40 Projects for Building Your Backyard Homeste...,2,Low
283,Simply Living Well: A Guide to Creating a Natu...,1,Low
284,Sustainable Badass: A Zero-Waste Lifestyle Gui...,4,Moderate
290,"Green from the Ground Up: Sustainable, Healthy...",2,Low
293,The Organic Country Home Handbook: How to Make...,2,Low
297,Zero Waste Home: The Ultimate Guide to Simplif...,3,Moderate
303,"Never Home Alone: From Microbes to Millipedes,...",1,Low
314,Back to Basics: A Complete Guide to Traditiona...,0,Low
316,"New Good Food Pocket Guide, rev: Shopper's Poc...",4,Moderate


In [41]:
df_analysis[
    df_analysis["Product_Type"] == "Possible Non-Product"
][
    ["Title","Sustainability_Score","Sustainability_Evidence_Level"]
]

,Title,Sustainability_Score,Sustainability_Evidence_Level
218,Waste-Free Kitchen Handbook: A Guide to Eating...,5,Moderate
281,40 Projects for Building Your Backyard Homeste...,2,Low
283,Simply Living Well: A Guide to Creating a Natu...,1,Low
284,Sustainable Badass: A Zero-Waste Lifestyle Gui...,4,Moderate
290,"Green from the Ground Up: Sustainable, Healthy...",2,Low
293,The Organic Country Home Handbook: How to Make...,2,Low
297,Zero Waste Home: The Ultimate Guide to Simplif...,3,Moderate
303,"Never Home Alone: From Microbes to Millipedes,...",1,Low
314,Back to Basics: A Complete Guide to Traditiona...,0,Low
316,"New Good Food Pocket Guide, rev: Shopper's Poc...",4,Moderate


## Sustainability Category

In [48]:
def classify_sustainability(row):
    score = row['Sustainability_Score']
    evidence = str(row['Sustainability_Evidence_Level']).strip().lower()

    if evidence == 'strong' and score >=6:
        return "Sustainable"
    elif(evidence == 'moderate' and score >=3) or (evidence == 'strong' and score < 6):
        return "Potentially Sustainable"
    else:
        return "Not Sustainable"
df_analysis["Sustainability_Category"] = df_analysis.apply(classify_sustainability, axis=1)

print(df_analysis[['Title',
         'Sustainability_Score',
         'Sustainability_Evidence_Level',
         'Sustainability_Category']].head(20))

                                                Title  Sustainability_Score  \
0   Plant-Based Kitchen Sponges - FSC Certified an...                    10   
1   Reusable Paper Towels Washable Roll - Thick 12...                     6   
2   20 Pack Reusable Paper Towels Washable - Natur...                     6   
3   Akeeko Reusable Beeswax Food Wraps - Assorted ...                    18   
4   Swedish Dishcloths for Kitchen Dish Towels - 6...                     7   
5   Lucomb Swedish Dishcloths for Kitchen Dishes, ...                     8   
6   Reusable Beeswax Wrap - 9 Pack Beeswax Wraps F...                    16   
7   100% Compostable Food Storage Bags [Quart 100 ...                     9   
8   Tieralia 5-Piece Eco-Friendly Bamboo Dish Brus...                     6   
9   Ultrasonic Pest Control Repeller - Repel Roden...                     2   
10  21 Pack Reusable Storage Bags BPA Free, Leak-p...                     6   
11  Swedish Dish Cloths - 10 Pack Reusable Kitchen..

In [49]:
print(df_analysis['Sustainability_Category'].value_counts())
print("\nPercentage:")
print(
    df_analysis['Sustainability_Category'].value_counts(normalize=True).mul(100).round(2)
)

Sustainability_Category
Not Sustainable            349
Sustainable                166
Potentially Sustainable    146
Name: count, dtype: int64

Percentage:
Sustainability_Category
Not Sustainable            52.80
Sustainable                25.11
Potentially Sustainable    22.09
Name: proportion, dtype: float64


## Price Vs Sustainability Analysis

In [55]:
price_analysis = df_analysis.groupby('Sustainability_Category')['Price'].agg(['count','mean','median','min','max']).round(2)
print(price_analysis)

                         count   mean  median   min     max
Sustainability_Category                                    
Not Sustainable            349  20.39   16.99  2.23  289.99
Potentially Sustainable    146  21.74   18.58  1.97   88.99
Sustainable                166  18.12   14.99  5.49   59.95


In [56]:
correlation = df_analysis['Price'].corr(df_analysis['Sustainability_Score'])
print("Price vs Sustainability Score Correlation:", round(correlation, 3))

Price vs Sustainability Score Correlation: -0.075


## Export Dataset

In [54]:
df_analysis.to_csv("Sustainable_Shopping_Analysis_Final.csv", index=False)